In [ ]:
!pip install datasets --quiet

In [ ]:
################################################################################

# Config

RUNTIME_TYPE = 'COLAB' #@param ["COLAB", "CONDA"]
MODEL_NAME = 't5-base' #@param ["t5-base", "google/flan-t5-xl", "t5-small", "t5-base", "t5-large", "google/flan-t5-small", "google/flan-t5-base", "google/flan-t5-large"]
LEARNING_RATE = 0.2 #@param {"type": "number"}
WD = 0 #@param {"type": "number"}

STUFF_COUNT = 2 #@param {"type": "number"}
PREDICT_WITH_GENERATE = False #@param {type:"boolean"}
DATASET_NAME = "TREC" #@param ["SST1", "SST2", "MR", "SUBJ", "CR", "MPQA", "TREC", "MPQA-P", "IMDB"]
MAX_LENGTHS_ARRAY = {"SST1": 256, "SST2": 256, "MR": 104,
                     "SUBJ": 217, "CR": 142, "MPQA": 58,
                     "TREC": 54}
SEED = 76 #@param {"type": "number"}
FOLDS = 10 #@param {"type": "number"}
MULTI_STEP_FOLDED = True #@param {type:"boolean"}
# STEP_FOLDS = [[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]]
# STEP_FOLDS = [[1, 2, 3, 4, 5], [6, 7, 8, 9, 10]]
STEP_FOLDS = [[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]]

STEP_INDICATOR = 0 #@param {"type": "number"}
HAS_TEST = ["SST1", "SST2", "TREC"]
NUM_TRAIN_EPOCHS = 60 #@param {"type": "number"}
LOSS_THRESH = 2 #@param {"type": "number"}
DO_K_FOLD = ["MPQA"]
BOUND_SUBSET_COUNT = 500 #@param {"type": "number"}
BOUND_SUBSET = [BOUND_SUBSET_COUNT]
EXPERIMENT_NAME = 'prompt-aug'
PROMPT_TOKEN_INDEX = 3
GENERATE_MAX_LEN = 128
FP16 = False
LOCAL_RANK = -1
FP16_OPT_LEVEL = 'O1'
BATCH_SIZE = 8
TRAIN_BATCH_SIZE = BATCH_SIZE
VAL_BATCH_SIZE = BATCH_SIZE
TEST_BATCH_SIZE = BATCH_SIZE
DROPOUT = 0.1
DATA = 'MPQA2.0_v221219_cleaned'
IDS = 'MPQA2.0_v221219_cleaned_Source_Span_IDs'
READ_DATA_ONLINE = False
SST2_DICT = {0: 'negative', 1: 'positive'}
IMDB_DICT = {0: 'negative', 1: 'positive'}
TREC_DICT = {0: 'Abbreviation', 1: 'Entity',
             2: 'Description', 3: 'Human being',
             4: 'Location', 5: 'Numeric value'}

SST2_COL_MAP = {'input': 'sentence', 'output': 'label'}
IMDB_COL_MAP = {'input': 'text', 'output': 'sentiment'}
TREC_COL_MAP = {'input': 'text', 'output': 'label'}

# Define special tokens for putting in the input sentence (prefix)
PREFIX = ['answer: ' ]
INIT_W = [None]
BEST_W = [None]
BEST_LOSS = [float('inf')]

INDICATOR = [0, 0]

INDICES_OUT = [list(), 0]


NUM_BEAMS = 2
LENGTH_PENALTY = 1.0
REPITITION_PENALTY = 2.5
DO_SAMPLE = False

MORE_THAN_ONE = {'exp': 0, 'target': 0, 'agent': 0}
TOTAL_COUNT = {'exp': 0, 'target': 0, 'agent': 0}


TOKENIZER = [None]
COUNTER = [0, 0]
SENTS_LENGHTS = dict()

ERRORS = []
ONCE_DONE = [False, False]
ACTUAL_TRIPLETS = [{}, {}]
GOLD_NUM = [None, None]

EVALUATION_CRI = [-1, 3]

WRITE_FILE = [False]

In [ ]:
import os
from os import chdir
import pandas as pd
import matplotlib.pyplot as plt
from copy import deepcopy
import re
import random
import numpy as np
import torch
from nltk import word_tokenize
import nltk
from nltk.corpus import stopwords
import re
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from urllib.request import urlopen
import urllib
import io
import transformers
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback, AutoConfig
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import json
from urllib.request import urlopen
import seaborn as sns
import statistics
from tqdm import tqdm
from torch import optim
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from datetime import datetime
import random
import time
from datetime import date
from datasets import load_dataset


from nltk.tokenize.treebank import TreebankWordDetokenizer
detokenizer = TreebankWordDetokenizer()

In [ ]:
# To assure deterministic results
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ":4096:8"
%matplotlib inline
%config InlineBackend.figure_format='retina'

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [ ]:
if RUNTIME_TYPE == 'COLAB':
  from google.colab import output
  output.enable_custom_widget_manager()
  from google.colab import drive
  drive.mount('/content/drive')
  if not os.path.exists(f'drive/MyDrive/{EXPERIMENT_NAME}'):
    os.makedirs(f'drive/MyDrive/{EXPERIMENT_NAME}')
  chdir(f'drive/MyDrive/{EXPERIMENT_NAME}')
  #os.system('rm -rf *')
else:
  if not os.path.exists(f'{EXPERIMENT_NAME}'):
    os.makedirs(f'{EXPERIMENT_NAME}')
  chdir(f'{EXPERIMENT_NAME}')

Mounted at /content/drive


In [ ]:
print(len(INDICES_OUT[0]))

0


In [ ]:
INDICES_OUT[1] = BOUND_SUBSET[0]

In [ ]:
!pwd

/content/drive/MyDrive/prompt-aug


In [ ]:
!nvidia-smi

Fri Jun 14 16:50:48 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   43C    P8               9W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
if RUNTIME_TYPE == 'COLAB':
    %pip install transformers[sentencepiece] --quiet
    %pip install torchinfo --quiet

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format='retina'

In [ ]:
# Start timer
start_time = datetime.now()

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f'There are {torch.cuda.device_count()} GPU(s) available.')
    print('Device name:', torch.cuda.get_device_name(0))

else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")


There are 1 GPU(s) available.
Device name: Tesla T4


TPI config

In [ ]:
TYPE_IDS = ['agreement', 'argue', 'intention', 'sentiment']
POLARITY_IDS = ['negative', 'positive']
INTENSITY_IDS = ['slight', 'low', 'medium', 'high', 'extreme']

In [ ]:
# TPI Classes

TYPE_CLASSES = ['agreement', 'arguing', 'intention', 'sentiment']
POLARITY_CLASSES = ['negative', 'positive']
INTENSITY_CLASSES = ['low', 'low medium', 'medium', 'medium high', 'high']
NUM_TYPE_CLASSES = len(TYPE_CLASSES)
NUM_POLARITY_CLASSES = len(POLARITY_CLASSES)
NUM_INTENSITY_CLASSES = 5
NUM_INTENSITY_UNITS = 3

if DATASET_NAME == 'MPQA-T':
    CLASSES = TYPE_CLASSES
    NUM_CLASSES = NUM_TYPE_CLASSES
elif DATASET_NAME == 'MPQA-P':
    CLASSES = POLARITY_CLASSES
    NUM_CLASSES = NUM_POLARITY_CLASSES
elif DATASET_NAME == 'MPQA-I':
    CLASSES = INTENSITY_CLASSES
    NUM_CLASSES = NUM_INTENSITY_CLASSES
else:
    CLASSES = []
    NUM_CLASSES = 0

In [ ]:
# Create a map for class ids and class names

type_classname2classindex = {
    'agreement': 0,
    'arguing':   1,
    'intention': 2,
    'sentiment': 3,
}
type_classname2classid = {
    'agreement': TYPE_IDS[0],
    'arguing':   TYPE_IDS[1],
    'intention': TYPE_IDS[2],
    'sentiment': TYPE_IDS[3],
}
type_classid2classname = {v:k for k, v in type_classname2classid.items()}
type_classid2classindex = {type_classname2classid[k]:v for k, v in type_classname2classindex.items()}

polarity_classname2classindex = {
    'negative': 0,
    'positive': 1,
}
polarity_classname2classid = {
    'negative': POLARITY_IDS[0],
    'positive': POLARITY_IDS[1],
}
polarity_classid2classname = {v:k for k, v in polarity_classname2classid.items()}
polarity_classid2classindex = {polarity_classname2classid[k]:v for k, v in polarity_classname2classindex.items()}

intensity_classname2classindices = {
    'low':        [0],
    'low medium': [0, 1],
    'medium':        [1],
    'medium high':   [1, 2],
    'high':             [2],
}
intensity_classname2classid = {
    'low':         INTENSITY_IDS[0],
    'low medium':  INTENSITY_IDS[1],
    'medium':      INTENSITY_IDS[2],
    'medium high': INTENSITY_IDS[3],
    'high':        INTENSITY_IDS[4],
}
intensity_classid2classname = {v:k for k, v in intensity_classname2classid.items()}
intensity_classid2classindices = {intensity_classname2classid[k]:v for k, v in intensity_classname2classindices.items()}

In [ ]:
import time
def pd_read_csv(link):
    t = 15
    while True:
        try:
            return pd.read_csv(link)
        except Exception as e:
            print(f"Unsuccessful attempt to download {link}. Waiting for {t}s.")
            time.sleep(t)
            t *= random.random()+1
            t = int(t)
            continue

In [ ]:
# Decompose X and y

def decompose_e2e(dataset):
    X = [i for i in dataset.mr]
    y = [i for i in dataset.ref]
    return X, y

def decompose_mpqa(dataset):
    X = [i for i in dataset.input]
    if DATASET_NAME == 'MPQA-T':
        y = [i.replace(TYPE_CLASSES[0], TYPE_IDS[0]) \
              .replace(TYPE_CLASSES[1], TYPE_IDS[1]) \
              .replace(TYPE_CLASSES[2], TYPE_IDS[2]) \
              .replace(TYPE_CLASSES[3], TYPE_IDS[3]) for i in dataset.output]
    elif DATASET_NAME == 'MPQA-P':
        y = [i.replace(POLARITY_CLASSES[0], POLARITY_IDS[0]) \
              .replace(POLARITY_CLASSES[1], POLARITY_IDS[1]) for i in dataset.output]
    elif DATASET_NAME == 'MPQA-I':
        # Add special characters to avoid conflicts
        y = [('!'+i+'!').replace('!'+INTENSITY_CLASSES[0]+'!', INTENSITY_IDS[0]) \
                        .replace('!'+INTENSITY_CLASSES[1]+'!', INTENSITY_IDS[1]) \
                        .replace('!'+INTENSITY_CLASSES[2]+'!', INTENSITY_IDS[2]) \
                        .replace('!'+INTENSITY_CLASSES[3]+'!', INTENSITY_IDS[3]) \
                        .replace('!'+INTENSITY_CLASSES[4]+'!', INTENSITY_IDS[4]) for i in dataset.output]
    return X, y

In [ ]:
# MPQA links
mpqa_t_train_link = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/t_train.csv'
mpqa_t_val_link   = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/t_val.csv'
mpqa_t_test_link  = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/t_test.csv'

mpqa_p_train_link = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/p_train.csv'
mpqa_p_val_link   = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/p_val.csv'
mpqa_p_test_link  = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/p_test.csv'

mpqa_i_train_link = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/i_train.csv'
mpqa_i_val_link   = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/i_val.csv'
mpqa_i_test_link  = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/i_test.csv'

In [ ]:
# Dataset public URL

data_name_to_google_drive_url = {
    'MPQA2.0_v221219_cleaned': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    'MPQA2.0_v221219_cleaned_Source_Span_IDs': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    '0.aaai19srl.train0.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '0.aaai19srl.dev0.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '0.aaai19srl.test0.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    '1.aaai19srl.train1.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '1.aaai19srl.dev1.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '1.aaai19srl.test1.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    '2.aaai19srl.train2.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '2.aaai19srl.dev2.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '2.aaai19srl.test2.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    '3.aaai19srl.train3.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '3.aaai19srl.dev3.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '3.aaai19srl.test3.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    '4.aaai19srl.train4.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '4.aaai19srl.dev4.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '4.aaai19srl.test4.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    'imdb-train': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    'imdb-test': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    'sst-2-train': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    'sst-2-dev': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    'sst-2-test': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    'trec-train': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    'trec-test': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    'all_zip': '<GOOGLE_DRIVE_URL_PLACEHOLDER>'
}

# Get direct download link
def get_download_url_from_google_drive_url(google_drive_url):
    return f'https://drive.google.com/uc?id={google_drive_url.split("/")[5]}&export=download&confirm=t'


In [ ]:
def set_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True

In [ ]:
set_seed()

In [ ]:
FILE_PATHS = {"SST1": ("data/stsa.fine.phrases.train",
                  "data/stsa.fine.dev",
                  "data/stsa.fine.test"),
              "SST2": ("data/stsa.binary.phrases.train",
                  "data/stsa.binary.dev",
                  "data/stsa.binary.test"),
              "MR": ("data/rt-polarity.all", "", ""),
              "SUBJ": ("data/subj.all", "", ""),
              "CR": ("data/custrev.all", "", ""),
              "MPQA": ("data/mpqa.all", "", ""),
              "TREC": ("data/TREC.train.all", "", "data/TREC.test.all"),
              }

In [ ]:
def clean_str(string):
  """
  Tokenization/string cleaning for all datasets except for SST.
  """
  string = re.sub(r"[^A-Za-z0-9(),!?\'\`]", " ", string)
  string = re.sub(r"\'s", " \'s", string)
  string = re.sub(r"\'ve", " \'ve", string)
  string = re.sub(r"n\'t", " n\'t", string)
  string = re.sub(r"\'re", " \'re", string)
  string = re.sub(r"\'d", " \'d", string)
  string = re.sub(r"\'ll", " \'ll", string)
  string = re.sub(r",", " , ", string)
  string = re.sub(r"!", " ! ", string)
  string = re.sub(r"\(", " ( ", string)
  string = re.sub(r"\)", " ) ", string)
  string = re.sub(r"\?", " ? ", string)
  string = re.sub(r"\s{2,}", " ", string)
  return string.strip().lower()

def clean_str_sst(string):
  """
  Tokenization/string cleaning for the SST dataset
  """
  string = re.sub(r"[^A-Za-z0-9(),!?\'\`]", " ", string)
  string = re.sub(r"\s{2,}", " ", string)
  return string.strip().lower()

In [ ]:
def line_to_words(line, dataset):
  if dataset == 'SST1' or dataset == 'SST2':
    clean_line = clean_str_sst(line.strip())
  else:
    clean_line = clean_str(line.strip())
  words = clean_line.split(' ')
  #words = words[1:]

  return words

In [ ]:
def load_data(dataset, train_name, test_name='', dev_name=''):
  """
  Load training data (dev/test optional).
  """

  # Initialize datasets
  train_dataset = dict({'input': [], 'output': []})
  dev_dataset = dict({'input': [], 'output': []})
  test_dataset = dict({'input': [], 'output': []})


  total_lines = list()

  f_train = open(train_name, 'r', encoding='ISO-8859-1')
  for line in f_train:
    words = line_to_words(line, dataset)
    input = str(' '.join(words[1:]))
    output = str(words[0])
    total_lines.append(line)
    train_dataset['input'].append(PREFIX[0] + input)
    train_dataset['output'].append(output)


  if not test_name == '':
    f_test = open(test_name, 'r', encoding='ISO-8859-1')
    for line in f_test:
      words = line_to_words(line, dataset)
      input = str(' '.join(words[1:]))
      output = str(words[0])
      total_lines.append(line)
      test_dataset[PREFIX[0] + input] = output
      test_dataset['input'].append(PREFIX[0] + input)
      test_dataset['output'].append(output)


  if not dev_name == '':
    f_dev = open(dev_name, 'r', encoding='ISO-8859-1')
    for line in f_dev:
      words = line_to_words(line, dataset)
      input = str(' '.join(words[1:]))
      output = str(words[0])
      total_lines.append(line)
      dev_dataset['input'].append(PREFIX[0] + input)
      dev_dataset['output'].append(output)


  print(len(total_lines))
  print(len(train_dataset['input']) + len(dev_dataset['input']) + len(test_dataset['input']))



  return train_dataset, dev_dataset, test_dataset, total_lines



In [ ]:
# Importing t5 models

from transformers import T5Tokenizer, T5ForConditionalGeneration, T5Model
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [ ]:
# Instantiate tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
TOKENIZER[0] = tokenizer


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
def get_encode_info(text, desired_token=':'):
  # get the token IDs of each word
  word_to_token = {}
  encoded = tokenizer.batch_encode_plus(
            [text],
            max_length=256,
            truncation=True,
            return_tensors="pt"
        )['input_ids'][0]
  #encoded = tokenizer.encode(text)
  for i, token_id in enumerate(encoded):
    token = tokenizer.decode([token_id])
    if token.startswith(' '):
        token = token[1:]
    if token.endswith(' '):
        token = token[:-1]
    if token not in word_to_token:
        word_to_token[token] = []
    word_to_token[token].append(i)

  return encoded[word_to_token[desired_token]]

In [ ]:
class SentimentDataset(Dataset):

  def __init__(self, texts, targets=None, max_len_in=64, max_len_out=32):
    self.texts = texts
    self.targets = targets
    self.tokenizer = TOKENIZER[0]
    self.max_len_in = max_len_in
    self.max_len_out = max_len_out

  def __len__(self):
    return len(self.texts)

  def __getitem__(self, item):
    text = self.texts[item]
    target = self.targets[item]

    encoding = self.tokenizer.batch_encode_plus(
            [text],
            max_length=self.max_len_in,
            pad_to_max_length=True,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

    target_encoding = self.tokenizer.batch_encode_plus(
            [target],
            max_length=self.max_len_out,
            pad_to_max_length=True,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )


    len_finding_encoding = tokenizer.batch_encode_plus(
          [text],
          truncation=True,
          return_tensors="pt",
    )

    last_len = len(len_finding_encoding['input_ids'][0])
    ######
    real_ids = encoding['input_ids']
    real_att = encoding['attention_mask']

    real_ids = real_ids[0, :]
    real_att = real_att[0, :]

    colon_token_id = get_encode_info(text)[0]

    colon_token_index = (real_ids == colon_token_id).nonzero()

    first_len = colon_token_index[0][0].item() + 1
    try:
      points = random.sample(range(first_len, last_len), STUFF_COUNT)
    except:
      print('Why')
      print(first_len)
      print(last_len)
      points = [first_len] * STUFF_COUNT

    points.sort()

    if INDICES_OUT[1] > 0:
      INDICES_OUT[0].append(points)
      INDICES_OUT[1] -= 1

    backward = 0

    for ii in range(len(points)):
        points[ii] = points[ii] + backward
        point = points[ii]
        tt = torch.full((1, 1), 60000)[0, :]
        aa = torch.full((1, 1), 1)[0, :]
        real_ids = torch.cat([real_ids[0: point], tt, real_ids[point: ]], 0)
        real_att = torch.cat([real_att[0: point], aa, real_att[point: ]], 0)
        backward += 1
        #print(real_ids)
        #print(real_att)
        #print('Iteration finished!', '\n')

    return {
    'text': text,
    'source_ids': real_ids,
    'source_mask': real_att,
    'target_ids': target_encoding['input_ids'].float().squeeze(),
    'target_mask': target_encoding['attention_mask'].float().squeeze(),
    'max_len_in': self.max_len_in,
    'points': torch.tensor(points)
    }

In [ ]:
def create_data_loader(texts, targets=None, max_len_in=64, max_len_out=8, batch_size=16):
  ds = SentimentDataset(
    texts=texts,
    targets=targets,
    max_len_in=max_len_in,
    max_len_out=max_len_out
  )

  return DataLoader(
    ds,
    batch_size=batch_size,
    num_workers=0
  )

In [ ]:
def initilize_model(freeze_embedding=False):

    model_hf = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
    #model_hf = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

    cnt = 0
    for n, p in model_hf.named_parameters():
        p.requires_grad = False

    return model_hf


In [ ]:
import torch.optim as optim
import torch.nn as nn

def load_optimizer(model, learning_rate=1e-5, weight_decay=0):

    #optimizer = optim.AdamW(optimizer_grouped_parameters, lr=learning_rate)

    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    return optimizer

In [ ]:
class SentimentClassifier(nn.Module):

  def __init__(self, t5model):
    super(SentimentClassifier, self).__init__()
    self.model = t5model

  def forward(self, ids, mask, y_ids=None, eval_time=False, max_len=64, points=None):

    if eval_time:
        outputs = self.model.generate(
                  input_ids = ids,
                  attention_mask = mask,
                  max_length=max_len,
            )
    else:
        outputs = self.model(input_ids=ids,
                attention_mask=mask,
                labels=y_ids)

    return outputs


In [ ]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

In [ ]:
def train_epoch(
  model,
  data_loaders,
  optimizer,
  device,
  n_examples,
  scheduler=None,
  epoch_num=1,
  eval_time=False
):

  if eval_time:
    model = model.eval()
  else:
    model = model.train()


  losses = []
  correct_predictions = 0
  f1 = 0

  INDICATOR = [0, 0]

  for data_loader in data_loaders:
    for d in data_loader:

        points = d["points"]

        outputs = model(d["source_ids"].to(device, dtype=torch.long),
                        d["source_mask"].to(device, dtype=torch.long),
                        y_ids=d["target_ids"].to(device, dtype=torch.long)
                        )
        loss = outputs.loss

        INDICATOR[0] += BATCH_SIZE

        #loss = outputs[0]
        #loss, prediction_scores = outputs[:2]

        if not eval_time:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()


        losses.append(loss.item())



  return np.mean(losses)

In [ ]:
def eval_model(model, data_loaders, device, n_examples):
  model = model.eval()

  losses = []
  correct_predictions = 0
  predictions = []
  actuals = []
  input_texts = []
  indices = []

  data_items = {}

  INDICATOR = [0, 0]

  for data_loader in data_loaders:
    with torch.no_grad():
      for d in data_loader:
        y = d['target_ids'].to(device, dtype = torch.long)
        ids = d['source_ids'].to(device, dtype = torch.long)
        mask = d['source_mask'].to(device, dtype = torch.long)
        input_text = d["text"]


        inputs_dict = dict.fromkeys(input_text, [])

        if PREDICT_WITH_GENERATE:

          generated_ids = model(
              ids,
              mask,
              eval_time=True,
              max_len=GENERATE_MAX_LEN)


          preds = [tokenizer.decode(g, skip_special_tokens=True, clean_up_tokenization_spaces=True) for g in generated_ids]
          target = [tokenizer.decode(t, skip_special_tokens=True, clean_up_tokenization_spaces=True)for t in y]
          predictions.extend(preds)
          actuals.extend(target)
          input_texts.extend(input_text)

        else:
          points = d["points"]

          outputs = model(d["source_ids"].to(device, dtype=torch.long),
                          d["source_mask"].to(device, dtype=torch.long),
                          y_ids=d["target_ids"].to(device, dtype=torch.long),
                          eval_time=True,
                          points=points
                          )
          loss = outputs.loss


        INDICATOR[0] += 1



        #losses.append(loss.item())

  return predictions, actuals, input_texts

In [ ]:
def clear_string(txt):
  txt = txt.lower()

  if txt.startswith('{'):
    txt = txt[1:]

  if txt.endswith('}'):
    txt = txt[:-1]

  txt = re.sub(':', ' ', txt)
  txt = re.sub('{', ' ', txt)
  txt = re.sub('}', ' ', txt)
  txt = re.sub(' = ', ' ', txt)
  txt = re.sub('=', ' ', txt)
  txt = re.sub('  ', ' ', txt)
  txt = re.sub(' ', '', txt)
  txt = re.sub('^ ', '', txt)
  txt = re.sub('|', '', txt)
  txt = re.sub(' $', '', txt)
  txt = re.sub('|$', '', txt)
  txt = re.sub('^|', '', txt)
  txt = re.sub('^:', '', txt)
  txt = re.sub(':$', '', txt)
  return txt

In [ ]:
def train_eval(model, train_data_loaders,
               train_set_size, epochs=4):

  history = defaultdict(list)
  best_accuracy = 0
  best_dev_loss = float('inf')
  save_all_data_v = {}
  save_all_data_t = {}

  # Start training loop
  print("Start training...\n")

  INIT_W[0] = deepcopy(model.model.encoder.embed_tokens)

  #print(model.model.encoder.embed_tokens.mid_learned_embedding[0][0])

  for epoch in range(epochs):
    if False:
      print(f'Epoch {epoch + 1}/{epochs}')
      print('-' * 20)

    start_time = time.time()

    loss = train_epoch(
      model,
      train_data_loaders,
      optimizer,
      device,
      train_set_size,
      epoch_num=epoch,
      eval_time=False
    )

    #print(f'Train loss {loss}')

    if loss < BEST_LOSS[0]:
      print(f'Updated: {BEST_LOSS[0]} --> {loss}, in epoch {epoch + 1}/{epochs}')
      BEST_LOSS[0] = loss
      BEST_W[0] = deepcopy(model.model.encoder.embed_tokens)

      #
      # eval_model(model, train_data_loaders, device, train_set_size)

    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)
    if False:
      print(f'Epoch Time: {epoch_mins}m {epoch_secs}s\n')

    if loss < LOSS_THRESH:
      break

  print(12 * ' = ')
  print('\n')
  return True

In [ ]:
def make_dataloaders(train, y_train):

  max_len_cnt_train = [MAX_LENGTHS_ARRAY[DATASET_NAME] + STUFF_COUNT]
  token_len_cnt_train = [0] * len(max_len_cnt_train)
  out_max_len_cnt_train = [0] * len(max_len_cnt_train)
  trains = []
  y_trains = []


  for i in range(len(max_len_cnt_train)):
    trains.append(list())
    y_trains.append(list())


  token_lens = []

  for k in range(len(train)):

    tokens1 = tokenizer(train[k])['input_ids']
    tokens2 = tokenizer(y_train[k])['input_ids']
    token_lens.append(max(len(tokens1), len(tokens2)))

    for i in range(len(max_len_cnt_train)):
      if max(len(tokens1), len(tokens2)) <= max_len_cnt_train[i]:
        token_len_cnt_train[i] += 1
        trains[i].append(train[k])
        y_trains[i].append(y_train[k])
        out_max_len_cnt_train[i] = max(out_max_len_cnt_train[i], len(tokens2))
        break

  #new method
  train_data_loaders = []


  for i in range(len(trains)):
    train_data_loader = create_data_loader(trains[i], targets=y_trains[i], max_len_in=max_len_cnt_train[i], max_len_out=out_max_len_cnt_train[i], batch_size=TRAIN_BATCH_SIZE)
    train_data_loaders.append(train_data_loader)

  print(max(token_lens))
  return train_data_loaders

In [ ]:
class SoftEmbedding(nn.Module):
    def __init__(self,
                wte: nn.Embedding,
                random_range: float = 0.5,
                initialize_from_vocab: bool = True):

        super(SoftEmbedding, self).__init__()
        self.wte = wte
        self.wte.requires_grad = False
        # self.learned_embedding = nn.parameter.Parameter(self.initialize_embedding(wte,
        #                                                                        random_range,
        #                                                                        initialize_from_vocab))


        params_list = []
        for i in range(BOUND_SUBSET[0]):
          for j in range(STUFF_COUNT):
            params_list.append(nn.parameter.Parameter(self.initialize_embedding(wte,
                                                                                1,
                                                                                random_range,
                                                                                initialize_from_vocab)))

        self.mid_learned_embedding = nn.ParameterList(params_list)


    def initialize_embedding(self,
                             wte: nn.Embedding,
                             n_tokens: int = 10,
                             random_range: float = 0.5,
                             initialize_from_vocab: bool = True):

        tt = torch.FloatTensor(n_tokens, wte.weight.size(1))

        sam = random.sample(range(0, wte.weight.size(0)), n_tokens)
        tt = self.wte.weight[sam].clone().detach()

        return tt

    def forward(self, tokens):


        #sentence_tokens = tokens[:, self.n_tokens:]
        sentence_tokens = tokens[:, :]
        sentences_embs = torch.full((tokens.shape[0], tokens.shape[1], self.wte.weight.size(1)), 0, dtype = torch.float).to(device)
        indices = [None] * sentence_tokens.shape[0]
        for i in range(sentence_tokens.shape[0]):
          backward = 0
          for j in range(sentence_tokens[i, :].shape[0]):
            if sentence_tokens[i, j] == 60000:
              if indices[i]:
                indices[i].append(j + backward)
              else:
                indices[i] = [j + backward]
              #backward += 1

          # The last of us
          indices[i].append(sentence_tokens[i, :].shape[0] + backward)

        #print(learned_embedding.shape)
        segments = []

        for j in range(tokens.shape[0]):
          i = j + INDICATOR[0]
          # For the general case
          # STUFF_COUNT gets involved
          input_embedding = []
          for k in range(STUFF_COUNT):
            input_embedding.append(self.mid_learned_embedding[STUFF_COUNT * i + k].to(device))

          # Now we should paste the parts together
          progress_list = [self.wte(sentence_tokens[i, 0: indices[i][0]])]

          for k in range(STUFF_COUNT):
            progress_list.append(input_embedding[k])
            current_segment = self.wte(sentence_tokens[i, indices[i][k] + 1: indices[i][k + 1]])
            progress_list.append(current_segment)

          # Build the tensor and put it in the right place
          final = torch.cat(progress_list, 0)
          sentences_embs[i, :, :] = final

        return sentences_embs

In [ ]:
def create_piped(sentence, annots):
    span = ''
    the_sentence = detokenizer.detokenize(sentence)
    the_span = ''
    the_target = ''
    the_agent = ''
    itself = {}
    span = ''
    agent = ''
    target = ''
    role_target = ''
    role_agent = ''
    role_exp = ''

    for item1 in annots:
        if item1[4] == 'DSE':
            sentence_prime = sentence.copy()
            the_agent = ' '
            the_target = ' '


            for item2 in annots:
                if item2[0] == item1[0] and item2[1] == item1[1]:
                    if item2[4] == 'TARGET':
                        the_target = detokenizer.detokenize(sentence[item2[2]: item2[3] + 1])

                    elif item2[4] == 'AGENT':
                        the_agent = detokenizer.detokenize(sentence[item2[2]: item2[3] + 1])




            the_span = detokenizer.detokenize(sentence[item1[0]: item1[1] + 1])

            role_exp = role_exp + ' | ' + the_span

    if role_exp.startswith(' | '):
        role_exp = role_exp[3:]

    if span.startswith(' | '):
        span = span[3:]


    role_exp = '{ ' + role_exp + ' }'

    sample = {'role_exp': role_exp}

    return sample

In [ ]:
def sentiment_tsv_data_reader(data_name, conversion_dict, col_map):
  google_drive_url = data_name_to_google_drive_url[data_name]
  data_url = get_download_url_from_google_drive_url(google_drive_url)
  response = urlopen(data_url)
  # Get path to file in Colab
  file_path = io.BytesIO(response.read())
  # Read TSV file into DataFrame
  df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip')
  # Extract data samples
  dataset = dict()
  print(df.columns)
  if BOUND_SUBSET[0] > -1:
      counter = 0

  for iterator in range(df.shape[0]):
    if BOUND_SUBSET[0] > -1:
          counter += 1
          if counter > BOUND_SUBSET[0]:
              break

    input = df.at[iterator, col_map['input']]
    output = df.at[iterator, col_map['output']]
    dataset[PREFIX[0] + input] = conversion_dict[output]

  return dataset


In [ ]:
def mpqa_data_reader(data_name):
  google_drive_url = data_name_to_google_drive_url[data_name]
  data_url = get_download_url_from_google_drive_url(google_drive_url)
  response = urlopen(data_url)
  dd = response.readlines()
  data = list()
  # Extract data samples
  dataset = dict()

  if BOUND_SUBSET[0] > -1:
      counter = 0

  for line in dd:
      if BOUND_SUBSET[0] > -1:
          counter += 1
          if counter > BOUND_SUBSET[0] or counter > len(dd):
              break

      data.append(json.loads(line.decode()))

  #
  for item in data:
    sentence = item['sentences']
    annots = item['orl']
    sentence_output = create_piped(sentence, annots)

    dataset[PREFIX[0] + detokenizer.detokenize(sentence)] = sentence_output['role_exp']

  return dataset

In [ ]:
def mpqa_tpi_data_reader(data_name):
  if data_name == 'MPQA-T':
      trainset = pd_read_csv(mpqa_t_train_link)
      valset   = pd_read_csv(mpqa_t_val_link)
      testset  = pd_read_csv(mpqa_t_test_link)
  elif data_name == 'MPQA-P':
      trainset = pd_read_csv(mpqa_p_train_link)
      valset   = pd_read_csv(mpqa_p_val_link)
      testset  = pd_read_csv(mpqa_p_test_link)
  elif data_name == 'MPQA-I':
      trainset = pd_read_csv(mpqa_i_train_link)
      valset   = pd_read_csv(mpqa_i_val_link)
      testset  = pd_read_csv(mpqa_i_test_link)

  X_train, y_train = decompose_mpqa(trainset)
  X_val,   y_val   = decompose_mpqa(valset)
  X_test,  y_test  = decompose_mpqa(testset)

  # Extract data samples
  dataset = dict()

  if BOUND_SUBSET[0] > -1:
      counter = 0

  for i in range(len(X_train)):
      if BOUND_SUBSET[0] > -1:
          counter += 1
          if counter > BOUND_SUBSET[0] or counter > len(X_train):
              break

      dataset[PREFIX[0] + X_train[i]] = y_train[i]


  return dataset, None, None

In [ ]:
def trec_data_reader(data_name, conversion_dict=None, col_map=None):
  trec_dataset = load_dataset('trec')
  train_dataset = trec_dataset[data_name]
  dataset = dict()
  if BOUND_SUBSET[0] > -1:
      counter = 0

  for item in train_dataset:
    if BOUND_SUBSET[0] > -1:
          counter += 1
          if counter > BOUND_SUBSET[0] or counter > train_dataset.shape[0]:
              break
    input = item['text']
    output = item['coarse_label']
    if PREFIX[0] + input in dataset:
      counter -= 1
    dataset[PREFIX[0] + input] = output if not conversion_dict else conversion_dict[output]

  # google_drive_url = data_name_to_google_drive_url[data_name]
  # data_url = get_download_url_from_google_drive_url(google_drive_url)
  # response = urlopen(data_url)
  # df = pd.read_json(response)

  # print(df.columns)
  # print(df.shape)
  # unique_values = df['label'].unique()
  # print(unique_values)

  # # Extract data samples
  # dataset = dict()

  # if BOUND_SUBSET[0] > -1:
  #     counter = 0

  # for iterator in range(df.shape[0]):
  #   if BOUND_SUBSET[0] > -1:
  #         counter += 1
  #         if counter > BOUND_SUBSET[0]:
  #             break

  #   input = df.at[iterator, col_map['input']]
  #   output = df.at[iterator, col_map['output']]
  #   if PREFIX[0] + input in dataset:
  #     print('Duplicate\n====\n')
  #     print(PREFIX[0] + input)
  #     print(output)
  #     print(12 * ' - ')
  #   dataset[PREFIX[0] + input] = output if not conversion_dict else conversion_dict[output]


  return dataset

In [ ]:
def shuffle_lists(list1, list2):
  # Combine the lists using zip()
  temp = list(zip(list1, list2))

  # Shuffle the zipped list
  random.shuffle(temp)

  # Unzip the shuffled list
  res1, res2 = zip(*temp)

  # Convert back to lists
  res1, res2 = list(res1), list(res2)

  return res1, res2

list1 = [1, 2, 3]
list2 = [11, 22, 33]
print(shuffle_lists(list1, list2))

([3, 1, 2], [33, 11, 22])


In [ ]:

INIT_W = [None]
BEST_W = [None]
BEST_LOSS = [float('inf')]

INDICATOR = [0, 0]

INDICES_OUT = [list(), BOUND_SUBSET[0]]

# Initialize datasets
train_dataset = dict({'input': [], 'output': []})
dev_dataset = dict({'input': [], 'output': []})
test_dataset = dict({'input': [], 'output': []})


#
if DATASET_NAME.find('MPQA-') >= 0:
  train_dataset, dev_dataset, test_dataset = mpqa_tpi_data_reader(DATASET_NAME)
else:

  if DATASET_NAME == 'IMDB':
    #
    train_dataset = sentiment_tsv_data_reader('imdb-train', IMDB_DICT, IMDB_COL_MAP)
    test_dataset = sentiment_tsv_data_reader('imdb-test', IMDB_DICT, IMDB_COL_MAP)

  else:
    data_name = 'all_zip'
    google_drive_url = data_name_to_google_drive_url[data_name]
    data_url = get_download_url_from_google_drive_url(google_drive_url)
    print(data_url)
    urllib.request.urlretrieve(data_url, "response.zip")
    os.system('unzip response.zip')
    train_path, dev_path, test_path = FILE_PATHS[DATASET_NAME]
    train_dataset, dev_dataset, test_dataset, total_lines = load_data(DATASET_NAME, train_path, test_name=test_path, dev_name=dev_path)




# Set seed
set_seed()

#
# Create a list of indices and shuffle it
indices = list(range(len(train_dataset['input'])))
random.shuffle(indices)

# Use the shuffled indices to rearrange your lists
train_dataset['input'] = [train_dataset['input'][i] for i in indices]
train_dataset['output'] = [train_dataset['output'][i] for i in indices]
#

###
# Assuming opt is a similar structure with attributes like 'folds', 'has_test', etc.
# Assuming all_train and all_train_label are PyTorch Tensors or similar structures like NumPy arrays
FOLD_ARRAY = []
if MULTI_STEP_FOLDED:
  folds_range = STEP_FOLDS[STEP_INDICATOR]
else:
  folds_range = range(1, FOLDS + 1)

folds_range = [1] #range(1, 11)
# Training folds.
for fold in folds_range:
    start_time = time.time()  # Start timer
    FOLD_ARRAY.append(fold)
    train = train_dataset['input']
    train_label = train_dataset['output']
    # train, train_label = shuffle_lists(train, train_label)
    test = test_dataset['input']
    test_label = test_dataset['output']

    print(train[ : 6])
    print(train_label[ : 6])
    print(set(train_label[ : 500]))
    INIT_W = [None]
    BEST_W = [None]
    BEST_LOSS = [float('inf')]

    INDICATOR = [0, 0]

    INDICES_OUT = [list(), BOUND_SUBSET[0]]
    print()
    print(f'==> fold {fold}')

    if DATASET_NAME not in HAS_TEST:
        # make train/test data (90/10 split for train/test)
        N = len(train)
        i_start = int((fold - 1) * (N / FOLDS))
        i_end = int(fold * (N / FOLDS))

        test = train[i_start:i_end]
        test_label = train_label[i_start:i_end]

        train = train[:i_start] + train[i_end:]
        train_label = train_label[:i_start] + train_label[i_end:]

    # shuffle train to get dev/train split (10% to dev)
    J = len(train)

    # Generate a shuffled array of indices from 0 to J-1
    shuffle = np.random.permutation(J)

    # Use the shuffled indices to reorder the train data and labels
    # Create an index list from 0 to J-1
    indexes = list(range(J))


    # Shuffle the index list
    random.shuffle(indexes)

    # Use the shuffled indexes to reorder the train data and labels using list comprehension
    train = [train[i] for i in indexes]
    train_label = [train_label[i] for i in indexes]


    num_batches = J // BATCH_SIZE

    num_train_batches = round(num_batches * 0.9)

    train_size = num_train_batches * BATCH_SIZE

    dev_size = J - train_size

    dev = train[train_size:]
    dev_label = train_label[train_size:]

    train = train[:train_size]
    train_label = train_label[:train_size]


    print('Total items in train data: ', len(train))


    ###
    # Select a subset of train set
    train = train[:BOUND_SUBSET[0]]
    train_label = train_label[:BOUND_SUBSET[0]]

    custom_train_dataset = dict({'input': [], 'output': []})
    custom_train_dataset['input'] = train
    custom_train_dataset['output'] = train_label


    train_data_loaders = make_dataloaders(train, train_label)

    print("Training set size:",   len(train_label))
    print("Dev set size:",   len(dev_label))
    print("Test set size:",   len(test_label))

    print('-'*10)


    model_hf = initilize_model(freeze_embedding=False)

    s_wte = SoftEmbedding(model_hf.get_input_embeddings(),
                          initialize_from_vocab=True)


    model_hf.encoder.set_input_embeddings(s_wte)

    model = SentimentClassifier(model_hf)

    #print(model.model.encoder.embed_tokens.wte.weight[32107, :])

    model = model.to(device)

    optimizer = load_optimizer(model, learning_rate=LEARNING_RATE, weight_decay=WD)


    _ = train_eval(model,
                  train_data_loaders,
                  len(train_label),
                  epochs=NUM_TRAIN_EPOCHS)


    out_dict = {'dataset': custom_train_dataset,
                'indices': INDICES_OUT[0],
                'batch_size': BATCH_SIZE,
                'stuff_count': STUFF_COUNT,
                'seed': SEED
                }
    # Save the new file
    with open(f'aug-train-{fold}-{SEED}.json', 'w', encoding='utf-8') as f:
        json.dump(out_dict, f, indent=4)
    #
    torch.save(BEST_W[0].mid_learned_embedding, f'mid_learned_embedding_{fold}_{SEED}.pth')

    fold_time = time.time() - start_time

    print(f'Time taken for fold {fold}: {fold_time} seconds')


    # if fold > 0:
    #   break

# Code of the experiment
FOLD_ARRAY = [str(item) for item in FOLD_ARRAY]
f_string = '_'.join(FOLD_ARRAY)
experiment_code = f'{DATASET_NAME}_{BOUND_SUBSET[0]}_SC_{STUFF_COUNT}_S_{SEED}_F_{f_string}'
# Zipping files into a single file
today = date.today()
formatted_date = today.strftime("%Y-%m-%d")
# os.system(f'tar -czvf {formatted_date}_{experiment_code}.tar.gz *')
#os.system(f'rm -rf data')
#os.system(f'rm -rf response.zip')

import tarfile
import datetime
# Create a tarball of all .pth and .json files
with tarfile.open(f"{formatted_date}_{experiment_code}.tar.gz", "w:gz") as tar:
    for file in os.listdir():
        if file.endswith(".pth") or file.endswith(".json"):
            tar.add(file)

os.system('find . -type f \( -name "aug-train-*.json" -o -name "mid_learned_embedding_*.pth" \) -delete')





#
print('\n* * * * *')
print('The End.')
print('* * * * *\n')
#